In [3]:
# 读取 /Users/misixuan/Desktop/codingwithai/rmsd_test/data/ready/t1x_dataset.pkl 文件
import numpy as np
import pickle
import os
import sys
from typing import Dict, List, Optional
from tqdm import tqdm

# 添加项目根目录到Python路径
import sys
sys.path.append("/Users/misixuan/Desktop/codingwithai/rmsd_test")

# 导入必要的模块
from src.utils import MolStructure, AtomMapping
from src.reorder_inertia_hungarian import reorder_inertia_hungarian

with open('/Users/misixuan/Desktop/codingwithai/rmsd_test/data/ready/t1x_dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

In [13]:
def generate_gjf_file(
    num: int,
    file_type: str,
    atom_symbols: list[str],
    coordinates: np.ndarray | list[list[float]]
) -> None:
    """
    生成GJF文件并保存到test文件夹
    
    参数:
        num: 编号（如9137）
        file_type: 文件类型，只能是 'react' 或 'prod'
        atom_symbols: 原子符号列表（如 ['C', 'O', ...]）
        coordinates: 坐标数组，形状为 (N, 3)，N为原子数
    """
    # 1. 校验输入参数
    if file_type not in ['react', 'prod']:
        raise ValueError("file_type 只能是 'react' 或 'prod'")
    if len(atom_symbols) != len(coordinates):
        raise ValueError("原子符号数量与坐标数量不匹配")
    
    # 2. 处理文件夹和文件名
    output_dir = "gjf"
    os.makedirs(output_dir, exist_ok=True)  # 不存在则创建test文件夹
    filename = f"{num}_{file_type}.gjf"
    output_path = os.path.join(output_dir, filename)
    
    # 3. 构建GJF文件内容
    gjf_content = []
    gjf_content.append("#")  # 第一行
    gjf_content.append("")  # 空行分隔
    gjf_content.append("Test")  # 标题行
    gjf_content.append("")  # 空行分隔
    gjf_content.append("0 1")  # 电荷和自旋多重度（默认0 1，可根据需求修改）
    
    # 添加原子和坐标（保留8位小数，与输入格式一致）
    for symbol, coord in zip(atom_symbols, coordinates):
        x, y, z = coord
        # 格式：元素 + 空格 + x坐标 + 空格 + y坐标 + 空格 + z坐标
        gjf_content.append(f"  {symbol}    {x:.8f}    {y:.8f}    {z:.8f}")
    
    # 最后添加一个空行（符合GJF格式要求）
    gjf_content.append("")
    
    # 4. 写入文件
    with open(output_path, "w", encoding="utf-8") as f:
        f.write("\n".join(gjf_content))
    
    print(f"GJF文件已生成：{output_path}")

In [5]:
manual_insp_no = 9137
print(dataset[manual_insp_no].structure_ref.atoms)
print(dataset[manual_insp_no].structure_cand.atoms)
print(dataset[manual_insp_no].mapping_indices + 1)
print(reorder_inertia_hungarian(dataset[manual_insp_no].structure_ref, dataset[manual_insp_no].structure_cand) + 1)


[6 8 6 6 6 6 6 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
[6 6 1 1 1 6 1 6 1 6 6 1 8 1 1 1 1 1 1 1 1]
[ 8 13 10 11  6  2  1  3  9 19 12 14 17 18 15  7 20 21  4 16  5]
[ 1 13  2 10  6 11  8 21  5 16 20  7 18 15 17 19  4 14  9  3 12]


In [14]:
generate_gjf_file(num=manual_insp_no, file_type='react', atom_symbols=dataset[manual_insp_no].structure_ref.get_atom_symbols(), coordinates=dataset[manual_insp_no].structure_ref.get_atom_coordinates())
generate_gjf_file(num=manual_insp_no, file_type='prod', atom_symbols=dataset[manual_insp_no].structure_cand.get_atom_symbols(), coordinates=dataset[manual_insp_no].structure_cand.get_atom_coordinates())


GJF文件已生成：gjf/9137_react.gjf
GJF文件已生成：gjf/9137_prod.gjf


In [60]:
from scripts.preprocess_data import create_test_atom_mapping

create_test_atom_mapping(
    ref_struct=dataset[0].structure_ref,
    target_struct=dataset[0].structure_cand,
    source=dataset[0].source,
    random_seed=42
).mapping_indices

array([0, 1, 5, 2, 4, 3, 6])

In [88]:
np.array_equal(dataset[0].structure_ref.atoms, dataset[0].structure_cand.atoms)

False

In [121]:
import numpy as np

def min_swap_count(correct_mapping, predicted_mapping):
    """
    计算两个排列数组（无重复元素）的最少交换次数
    前提：correct_mapping 和 predicted_mapping 是同一组元素的不同排列
    """
    # 1. 验证是否为排列关系（元素完全一致）
    if not np.array_equal(np.sort(correct_mapping), np.sort(predicted_mapping)):
        raise ValueError("两个数组不是排列关系，无法通过交换使它们一致")
    
    # 2. 构建置换映射 P：P[i] 表示 predicted_mapping[i] 在 correct_mapping 中的索引
    # （无重复元素时，用 argsort + searchsorted 高效映射）
    sorted_correct = np.sort(correct_mapping)
    correct_idx = np.argsort(correct_mapping)  # 记录 sorted_correct 中元素在原 correct 的索引
    predicted_sorted_idx = np.searchsorted(sorted_correct, predicted_mapping)  # 预测元素在 sorted_correct 中的位置
    permutation = correct_idx[predicted_sorted_idx]  # 置换数组
    
    # 3. 循环分解，统计循环个数
    n = len(correct_mapping)
    visited = np.zeros(n, dtype=bool)
    cycle_count = 0
    
    for i in range(n):
        if not visited[i]:
            cycle_count += 1
            j = i
            # 遍历当前循环的所有元素
            while not visited[j]:
                visited[j] = True
                j = permutation[j]  # 下一个元素的索引
    
    # 最少交换次数 = 数组长度 - 循环个数
    return n - cycle_count

In [127]:
min_swap_count(np.array([1, 2, 3, 4, 5]), np.array([5, 4, 3, 2, 1]))

2

In [130]:
np.sum(np.array([1, 2, 3, 4, 5]) != np.array([5, 4, 3, 2, 1]))

np.int64(4)